# 02 — Multi-Horizon Forecasting & Peak-Weighted Evaluation

**Objectives:**
1. Evaluate forecast accuracy at horizons h+1 through h+6
2. Introduce peak-weighted MAPE as the primary metric for BESS applications
3. Assess mock forecast temperature (Tx leads) as a feature

**Dataset:** North grid, 2017–2022  
**Models:** Random Forest, LightGBM (Linear Regression dropped — behaviour well understood from 01)  
**Test set:** Last 365 days (2021-07-02 → 2022-07-01)

## 0. Setup

In [ ]:
import subprocess
subprocess.run(['git', 'clone', 'https://github.com/mcyc/predictive-ml.git'])

import sys
sys.path.append('/kaggle/working/predictive-ml/load-forecast/src/')

In [ ]:
from data import load_data, split_chronological
from features import build_features, FEATURES, TARGET, add_tx_leads
from evaluate import (
    evaluate, peak_weighted_evaluate, compare_models,
    build_results_df, plot_residuals, plot_residuals_by_time
)
from models import train_rf, train_lgbm, get_feature_importance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('All imports successful.')

In [ ]:
DATA_PATH  = '/kaggle/input/<your-dataset>/your_data_file.parquet'  # <-- update
TEST_DAYS  = 365
HORIZONS   = [1, 2, 3, 4, 5, 6]  # hours ahead
PEAK_THRESHOLD = 0.90             # top 10% of load = peak

## 1. Load & Build Features

In [ ]:
df_raw = load_data(DATA_PATH)
df     = build_features(df_raw)
print(df.shape)
df.head()

## 2. Baseline: h+1 with Full Feature Set

Reproduce the h+1 result from notebook 01 as a sanity check before extending horizons.

In [ ]:
X_train, X_test, y_train, y_test = split_chronological(
    df, features=FEATURES, target=TARGET, test_days=TEST_DAYS
)

rf_h1   = train_rf(X_train, y_train)
lgbm_h1 = train_lgbm(X_train, y_train, X_test, y_test)

results_h1 = []
evaluate('RF h+1',   y_test, rf_h1.predict(X_test),   results=results_h1)
evaluate('LGBM h+1', y_test, lgbm_h1.predict(X_test), results=results_h1)
compare_models(results_h1)

In [ ]:
# Peak-weighted baseline
peak_results_h1 = []
peak_weighted_evaluate('RF h+1',   y_test, rf_h1.predict(X_test),
                       threshold=PEAK_THRESHOLD, results=peak_results_h1)
peak_weighted_evaluate('LGBM h+1', y_test, lgbm_h1.predict(X_test),
                       threshold=PEAK_THRESHOLD, results=peak_results_h1)
compare_models(peak_results_h1, peak=True)

## 3. Multi-Horizon Experiment

For each horizon h, the target is `north_clean` shifted back by h hours.
Lag features are adjusted so that `lag_1h` becomes the load at t-(h),
i.e. the most recent observation available at forecast time.

**Note:** For h > 1, `lag_1h` refers to the load h hours ago, not 1 hour ago.
We retrain separate models per horizon rather than chaining predictions,
which gives a cleaner upper bound on achievable accuracy at each horizon.

In [ ]:
# TODO: implement multi-horizon loop
# For each horizon h in HORIZONS:
#   1. Shift target back by h: y = df['north_clean'].shift(-h)
#   2. Adjust lag features so minimum lag = h (no leakage)
#   3. Train RF and LightGBM
#   4. Evaluate global + peak-weighted metrics
#   5. Store results keyed by horizon
# Then plot MAPE and peak-MAPE vs horizon

## 4. Mock Forecast Temperature (Tx Leads)

Add Tx lead features as a proxy for weather forecast temperature.
Compare h+1 accuracy with and without Tx_lead_1h to assess
whether forecast temperature is worth pursuing in production.

In [ ]:
# TODO: implement Tx leads experiment
# 1. Rebuild features with drop_na=False
# 2. Add Tx leads for horizons 1-6
# 3. Drop NaN and re-split
# 4. For each horizon h, compare model with and without Tx_lead_{h}h
# 5. Plot delta MAPE vs horizon

## 5. Results Summary

In [ ]:
# TODO: plot MAPE and peak-MAPE vs horizon for RF and LightGBM
# with and without Tx leads